In [1]:
#  CLASSIFICATION MODEL TRAINING
#  Project : Crop Yield Prediction Using Environmental and Nutrient Factors
#  Course  : Data Warehousing and Data Mining (DWDM)
#  Target  : Yield_Category  (Low=0 / Medium=1 / High=2)
#
#  Models trained separately :
#    1. Logistic Regression  (baseline — linear boundary)
#    2. Decision Tree        (non-linear, interpretable)
#    3. Naive Bayes          (probabilistic, fast)
#    4. Random Forest        (ensemble, best performer)


import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.utils import resample
from sklearn.metrics import (accuracy_score, f1_score,precision_score, recall_score,classification_report, confusion_matrix)
from sklearn.model_selection import cross_val_score, StratifiedKFold

# LOAD DATA

train = pd.read_csv("crop_yield_train_preprocessed.csv")
test  = pd.read_csv("crop_yield_test_preprocessed.csv")

FEAT_COLS = [c for c in train.columns if c not in ["Crop_Yield_kg_ha", "Yield_Category"]]
LABEL_MAP = {0: "Low", 1: "Medium", 2: "High"}


X_train_base = train[FEAT_COLS].values
X_test_base  = test[FEAT_COLS].values


X_train = np.hstack([X_train_base, train["Crop_Yield_kg_ha"].values.reshape(-1, 1)])
X_test  = np.hstack([X_test_base,  test["Crop_Yield_kg_ha"].values.reshape(-1, 1)])

y_train = train["Yield_Category"].values
y_test  = test["Yield_Category"].values


sc = StandardScaler()
X_train_lr = sc.fit_transform(X_train)
X_test_lr  = sc.transform(X_test)

In [2]:
# HELPERS

def print_scores(y_tr, y_tr_pred, y_te, y_te_pred):
    tr_acc = accuracy_score(y_tr, y_tr_pred)
    te_acc = accuracy_score(y_te, y_te_pred)
    te_f1  = f1_score(y_te, y_te_pred, average="macro")
    te_pre = precision_score(y_te, y_te_pred, average="macro")
    te_rec = recall_score(y_te, y_te_pred, average="macro")
    gap    = abs(tr_acc - te_acc)
    fit_status = (
        "Good fit"   if gap < 0.05 else
        "Overfit"   if tr_acc > te_acc else
        "Underfit"
    )
    return tr_acc, te_acc, te_f1

def print_report(y_te, y_te_pred):
    print(f"\n  Classification Report :")
    print(classification_report(y_te, y_te_pred,target_names=["Low", "Medium", "High"],digits=4))

def print_confusion(y_te, y_te_pred):
    cm = confusion_matrix(y_te, y_te_pred)
    print(f"  Confusion Matrix :")
    for i, label in enumerate(["Actual Low ", "Actual Med ", "Actual High"]):
        row_str = "  ".join(f"{cm[i, j]:>10,}" for j in range(3))
        print(f"  {label:<14} {row_str}")
    diag = cm.diagonal().sum()
    print(f"\n  Correctly classified : {diag:,} / {cm.sum():,}  ({diag/cm.sum()*100:.2f}%)")

def print_cv(model, X, y, cv=5, sample=50000):
    print(f"\n  5-Fold Stratified Cross-Validation  (sample = {min(sample,len(X)):,} rows)")
    n = min(sample, len(X))
    skf    = StratifiedKFold(n_splits=cv, shuffle=True, random_state=42)
    scores  = cross_val_score(model, X[:n], y[:n], cv=skf, scoring="accuracy", n_jobs=1)
    f1s     = cross_val_score(model, X[:n], y[:n], cv=skf, scoring="f1_macro",  n_jobs=1)
    print(f"  Accuracy per fold : {[round(s, 4) for s in scores]}")
    print(f"  Mean Accuracy     : {scores.mean():.4f}")
    print(f"  Std  Accuracy     : {scores.std():.4f}  {'(stable)' if scores.std() < 0.02 else '(unstable)'}")
    print(f"  Mean F1 (macro)   : {f1s.mean():.4f}")
    return scores, f1s

In [3]:
summary = []

#  MODEL 1 — LOGISTIC REGRESSION

m1 = LogisticRegression(C=1.0, solver="saga", max_iter=500,n_jobs=-1, random_state=42)

X_lr_s, y_lr_s = resample(X_train_lr, y_train, n_samples=50000,stratify=y_train, random_state=42)
print("Training...")
m1.fit(X_lr_s, y_lr_s)

tr_pred1 = m1.predict(X_lr_s)
te_pred1 = m1.predict(X_test_lr)
print("Training complete")

tr_acc1, te_acc1, te_f1_1 = print_scores(y_lr_s, tr_pred1, y_test, te_pred1)
print_report(y_test, te_pred1)
print_confusion(y_test, te_pred1)

cv_acc_1, cv_f1_1 = print_cv(
    LogisticRegression(C=1.0, solver="saga", max_iter=500, n_jobs=-1, random_state=42),
    X_train_lr, y_train, sample=30000)

summary.append(["Logistic Regression", tr_acc1, te_acc1, te_f1_1, cv_acc_1.mean(), cv_acc_1.std(), cv_f1_1.mean()])

Training...
Training complete

  Classification Report :
              precision    recall  f1-score   support

         Low     0.8922    0.9595    0.9246     14520
      Medium     0.8969    0.8336    0.8641     14520
        High     0.9505    0.9461    0.9483     14960

    accuracy                         0.9134     44000
   macro avg     0.9132    0.9131    0.9123     44000
weighted avg     0.9136    0.9134    0.9127     44000

  Confusion Matrix :
  Actual Low         13,932         588           0
  Actual Med          1,679      12,104         737
  Actual High             4         803      14,153

  Correctly classified : 40,189 / 44,000  (91.34%)

  5-Fold Stratified Cross-Validation  (sample = 30,000 rows)
  Accuracy per fold : [np.float64(0.8712), np.float64(0.8647), np.float64(0.8727), np.float64(0.8655), np.float64(0.8682)]
  Mean Accuracy     : 0.8684
  Std  Accuracy     : 0.0031  (stable)
  Mean F1 (macro)   : 0.8644


In [4]:
#  MODEL 2 — DECISION TREE CLASSIFIER


m2 = DecisionTreeClassifier(
    max_depth        = 12,
    min_samples_leaf = 10,
    min_samples_split= 20,
    criterion        = "gini",
    class_weight     = "balanced",
    random_state     = 42
)
print("Training...")
m2.fit(X_train, y_train)

tr_pred2 = m2.predict(X_train)
te_pred2 = m2.predict(X_test)
print("Training complete")

tr_acc2, te_acc2, te_f1_2 = print_scores(y_train, tr_pred2, y_test, te_pred2)
print_report(y_test, te_pred2)
print_confusion(y_test, te_pred2)

cv_acc_2, cv_f1_2 = print_cv(
    DecisionTreeClassifier(max_depth=12, min_samples_leaf=10,min_samples_split=20, criterion="gini",class_weight="balanced", random_state=42),
    X_train, y_train)

summary.append(["Decision Tree", tr_acc2, te_acc2, te_f1_2,cv_acc_2.mean(), cv_acc_2.std(), cv_f1_2.mean()])

Training...
Training complete

  Classification Report :
              precision    recall  f1-score   support

         Low     0.8971    0.6994    0.7860     14520
      Medium     0.9989    0.4982    0.6648     14520
        High     0.5777    0.9824    0.7276     14960

    accuracy                         0.7292     44000
   macro avg     0.8246    0.7266    0.7261     44000
weighted avg     0.8221    0.7292    0.7261     44000

  Confusion Matrix :
  Actual Low         10,155           2       4,363
  Actual Med            907       7,234       6,379
  Actual High           258           6      14,696

  Correctly classified : 32,085 / 44,000  (72.92%)

  5-Fold Stratified Cross-Validation  (sample = 50,000 rows)
  Accuracy per fold : [np.float64(0.7311), np.float64(0.7257), np.float64(0.7317), np.float64(0.7282), np.float64(0.7284)]
  Mean Accuracy     : 0.7290
  Std  Accuracy     : 0.0022  (stable)
  Mean F1 (macro)   : 0.7256


In [5]:
#  MODEL 3 — NAIVE BAYES


m3 = GaussianNB(var_smoothing=1e-9)
print("Training...")
m3.fit(X_train, y_train)

tr_pred3 = m3.predict(X_train)
te_pred3 = m3.predict(X_test)
print("Training complete")

tr_acc3, te_acc3, te_f1_3 = print_scores(y_train, tr_pred3, y_test, te_pred3)
print_report(y_test, te_pred3)
print_confusion(y_test, te_pred3)

cv_acc_3, cv_f1_3 = print_cv(GaussianNB(), X_train, y_train)

summary.append(["Naive Bayes", tr_acc3, te_acc3, te_f1_3, cv_acc_3.mean(), cv_acc_3.std(), cv_f1_3.mean()])

Training...
Training complete

  Classification Report :
              precision    recall  f1-score   support

         Low     0.5110    0.6683    0.5792     14520
      Medium     0.3828    0.1364    0.2011     14520
        High     0.5199    0.6894    0.5928     14960

    accuracy                         0.5000     44000
   macro avg     0.4712    0.4980    0.4577     44000
weighted avg     0.4717    0.5000    0.4590     44000

  Confusion Matrix :
  Actual Low          9,704       1,679       3,137
  Actual Med          6,154       1,980       6,386
  Actual High         3,133       1,513      10,314

  Correctly classified : 21,998 / 44,000  (50.00%)

  5-Fold Stratified Cross-Validation  (sample = 50,000 rows)
  Accuracy per fold : [np.float64(0.499), np.float64(0.4974), np.float64(0.4985), np.float64(0.5007), np.float64(0.4971)]
  Mean Accuracy     : 0.4985
  Std  Accuracy     : 0.0013  (stable)
  Mean F1 (macro)   : 0.4521


In [6]:
#  MODEL 4 — RANDOM FOREST CLASSIFIER

m4 = RandomForestClassifier(
    n_estimators    = 100,
    max_depth       = 15,
    min_samples_leaf= 10,
    max_features    = "sqrt",
    class_weight    = "balanced",
    n_jobs          = -1,
    random_state    = 42
)
print("Training...  ")
m4.fit(X_train, y_train)

tr_pred4 = m4.predict(X_train)
te_pred4 = m4.predict(X_test)
print("Training complete")

tr_acc4, te_acc4, te_f1_4 = print_scores(y_train, tr_pred4, y_test, te_pred4)
print_report(y_test, te_pred4)
print_confusion(y_test, te_pred4)

cv_acc_4, cv_f1_4 = print_cv(
    RandomForestClassifier(n_estimators=30, max_depth=15,min_samples_leaf=10, max_features="sqrt",n_jobs=-1, random_state=42),
    X_train, y_train)

summary.append(["Random Forest", tr_acc4, te_acc4, te_f1_4, cv_acc_4.mean(), cv_acc_4.std(), cv_f1_4.mean()])

Training...  
Training complete

  Classification Report :
              precision    recall  f1-score   support

         Low     0.9634    0.9220    0.9423     14520
      Medium     0.8939    0.8829    0.8884     14520
        High     0.8971    0.9453    0.9205     14960

    accuracy                         0.9170     44000
   macro avg     0.9182    0.9167    0.9171     44000
weighted avg     0.9180    0.9170    0.9171     44000

  Confusion Matrix :
  Actual Low         13,388         728         404
  Actual Med            482      12,820       1,218
  Actual High            26         793      14,141

  Correctly classified : 40,349 / 44,000  (91.70%)

  5-Fold Stratified Cross-Validation  (sample = 50,000 rows)
  Accuracy per fold : [np.float64(0.8456), np.float64(0.8569), np.float64(0.8349), np.float64(0.8393), np.float64(0.8568)]
  Mean Accuracy     : 0.8467
  Std  Accuracy     : 0.0090  (stable)
  Mean F1 (macro)   : 0.8458


In [7]:
#  FINAL COMPARISON TABLE


print(f"\n  {'Model':<22} {'Train Acc':>10} {'Test Acc':>9} {'F1-macro':>9}"
      f" {'CV Acc':>9} {'CV Std':>8} {'Status'}")


best_model = ""
best_acc   = -1
for row in summary:
    name, tr_a, te_a, f1, cv_m, cv_s, cv_f1 = row
    gap    = abs(tr_a - te_a)
    status = "Good fit" if gap < 0.05 else "Overfit" if tr_a > te_a else "Underfit"
    print(f"  {name:<22} {tr_a:>10.4f} {te_a:>9.4f} {f1:>9.4f}"
          f" {cv_m:>9.4f} {cv_s:>8.4f} {status}")
    if te_a > best_acc:
        best_acc   = te_a
        best_model = name

print(f"""
  Conclusion :
    Best model    → {best_model}
    Test Accuracy → {best_acc:.4f}""")


  Model                   Train Acc  Test Acc  F1-macro    CV Acc   CV Std Status
  Logistic Regression        0.9184    0.9134    0.9123    0.8684   0.0031 Good fit
  Decision Tree              0.7297    0.7292    0.7261    0.7290   0.0022 Good fit
  Naive Bayes                0.4992    0.5000    0.4577    0.4985   0.0013 Good fit
  Random Forest              0.9463    0.9170    0.9171    0.8467   0.0090 Good fit

  Conclusion :
    Best model    → Random Forest
    Test Accuracy → 0.9170
